# Exploración + filtrado (Ktalyna) y ampliación a 201 genes

**MVA Hackathon 2026 · Track 2 — notebook de estudio de Laura**

Reúne en un solo lugar:
- **Parte 1:** la exploración inicial de Ktalyna (`notebooks/01_exploracion_inicial.ipynb`).
- **Parte 2:** el filtrado por 6 genes de Ktalyna (`notebooks/02_filtrado_vcf_genes_mva.ipynb`).
- **Parte 3:** la ampliación a la lista del equipo (201 genes).

Cada celda indica de quién es: **[Ktalyna]**, **[Agregado]** (revisiones o ayudas) o **[Ampliación]**.

> **Cómo correrlo:** Colab conectado a tu PC ("entorno de ejecución local"), con Jupyter iniciado desde `C:\dev\mva-hackathon-2026`. Corre las celdas en orden. Las celdas 4, 8 y 15 recorren el VCF completo y tardan 1–3 minutos cada una.

---

### El embudo

```
5,012,204  variantes en el VCF
    │  Paso 1: sello PASS         ← lo puso el LABORATORIO; aquí solo se lee
4,740,790
    │  Paso 2: ¿cae en un gen candidato?   ← trabajo de Ktalyna (6 genes) / ampliación (201 genes)
1,075  (6 genes)   |   ~19,650  (201 genes)
    │  Pasos 3–5: rareza, impacto, dos copias afectadas   ← notebook 04 (siguiente)
```

## PARTE 1 — Exploración inicial [Ktalyna]

In [ ]:
# Celda 0 · [Agregado] Ubicarse en la carpeta del proyecto
# El original usaba Path("../data/raw"), que solo funciona si el notebook corre desde notebooks/.
# Esta celda busca la raíz del proyecto.
import os
from pathlib import Path

root = Path.cwd()
while not (root / "scripts").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("Carpeta de trabajo:", Path.cwd())

In [ ]:
# Celda 1 · [Ktalyna] Verificar el entorno
# Comprueba la sesión en Hugging Face y que los 3 archivos estén descargados.
import sys
from huggingface_hub import whoami

print(f"Python: {sys.version}")
try:
    info = whoami()
    print(f"HuggingFace: autenticada como {info['name']} ✓")
except Exception:
    print("⚠️  No autenticada en HuggingFace. Ejecuta: python scripts/hf_login.py")

DATA_DIR = Path("data/raw")
expected = [
    "Challenge_Clinical_Phenotype_1.docx",
    "WGS_EX2312012_HGWCNDSX7.vcf.gz",
    "WGS_EX2312012_HGWCNDSX7.vcf.gz.tbi",
]
for fname in expected:
    p = DATA_DIR / fname
    status = f"{p.stat().st_size / 1e6:.1f} MB ✓" if p.exists() else "❌ NO ENCONTRADO"
    print(f"  {fname}: {status}")

In [ ]:
# Celda 2 · [Ktalyna] Fenotipo del paciente (HPO)
# ⚠️ Información real y confidencial del paciente.
# Salida esperada: 8 filas (rabdomiosarcoma, nefrocalcinosis, talla baja, falla de crecimiento,
# atrofia muscular, prematuridad, bajo peso al nacer, abortos recurrentes de los padres).
from docx import Document
import pandas as pd

doc = Document(DATA_DIR / "Challenge_Clinical_Phenotype_1.docx")
table = doc.tables[0]
rows = [[cell.text.strip() for cell in row.cells] for row in table.rows]
df_hpo = pd.DataFrame(rows[1:], columns=rows[0])
df_hpo

In [ ]:
# Celda 3 · [Ktalyna] Encabezado del VCF
# Qué mirar: VCFv4.2; Sentieon Haplotyper; filtros GATK; referencia GRCh38 sin prefijo chr;
# una sola muestra (WGS_EX2312012).
import gzip

VCF = DATA_DIR / "WGS_EX2312012_HGWCNDSX7.vcf.gz"

meta_lines = []
with gzip.open(VCF, "rt") as f:
    for line in f:
        if not line.startswith("#"):
            break
        meta_lines.append(line.strip())

important_tags = ("fileformat", "GATKCommand", "SentieonCommand", "reference", "#CHROM")
for l in meta_lines:
    if any(tag in l for tag in important_tags):
        print(l[:200])

In [ ]:
# Celda 3b · [Agregado] Una línea del VCF, columna por columna
# Salida esperada: una fila por variante, con cromosoma, posicion, rsID,
# alelos, filtro de calidad y genotipo. Los valores del paciente no se
# versionan (convencion de privacidad del equipo).
# → heterocigota, 24 lecturas con C y 17 con T.
with gzip.open(VCF, "rt") as f:
    for line in f:
        if not line.startswith("#") and line.split("\t")[6] == "PASS":
            break
cols = line.rstrip("\n").split("\t")
names = ["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT", "MUESTRA"]
for n, v in zip(names, cols):
    print(f"{n:8s} = {v[:80]}")

In [ ]:
# Celda 4 · [Ktalyna] Estadísticas del VCF ⏱️ 1–3 min
# Salida esperada: 5,012,204 total · 4,740,790 PASS (94.6%) · ~81% SNV / ~19% INDEL.
# Nota: las 271,414 no-PASS las marcó el laboratorio (principalmente MQ40: lecturas mal
# ubicadas en zonas repetidas). Pendiente: análisis de sensibilidad sobre no-PASS en genes candidatos.
from collections import Counter
import matplotlib.pyplot as plt

filters = Counter()
variant_types = Counter()
chroms = Counter()

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.split("\t", 8)
        ref, alt, filt = parts[3], parts[4], parts[6]
        chrom = parts[0]
        filters[filt] += 1
        variant_types["SNV" if len(ref) == 1 and len(alt) == 1 else "INDEL"] += 1
        chroms[chrom] += 1

total = sum(filters.values())
print(f"Total variantes: {total:,}")
print(f"  PASS: {filters['PASS']:,} ({filters['PASS']/total*100:.1f}%)")
print(f"  Filtradas: {total - filters['PASS']:,} ({(total-filters['PASS'])/total*100:.1f}%)")
print("\nPor tipo:")
for k, v in variant_types.items():
    print(f"  {k}: {v:,} ({v/total*100:.1f}%)")

In [ ]:
# Celda 5 · [Ktalyna] Gráficos
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
top_filters = dict(filters.most_common(7))
axes[0].barh(list(top_filters.keys()), list(top_filters.values()), color="steelblue")
axes[0].set_title("Variantes por filtro GATK")
axes[0].set_xlabel("Número de variantes")
axes[1].pie(list(variant_types.values()), labels=list(variant_types.keys()),
            autopct="%1.1f%%", colors=["#4C72B0", "#DD8452"])
axes[1].set_title("SNV vs INDEL")
plt.tight_layout()
plt.savefig("results/vcf_stats_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## PARTE 2 — Filtrado por 6 genes [Ktalyna]

Pregunta: de las 4,7 millones de variantes PASS, ¿cuáles caen en genes de MVA y en qué estado (una copia o las dos)?

| Gen | Por qué |
|-----|---------|
| BUB1B, CEP57, TRIP13 | Genes conocidos de MVA (tipos 1, 2, 3) |
| BUB1, MAD1L1, MAD2L1 | Otras piezas del freno de la división celular (SAC) |

In [ ]:
# Celda 6 · [Ktalyna] Imports
import json
import requests
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Celda 7 · [Ktalyna] Coordenadas GRCh38 de los 6 genes
# PADDING = 5000: se agregan 5.000 pb a cada lado para incluir el promotor.
# Salida esperada: MAD1L1 mide ~417 kb (el más largo), BUB1B ~60 kb.
FALLBACK_COORDS = {
    "BUB1B":  ("15", 40_160_984,  40_221_137),
    "CEP57":  ("11", 95_789_965,  95_837_070),
    "TRIP13": ("5",    892_849,      919_357),
    "BUB1":   ("2",  110_143_783, 110_244_769),
    "MAD1L1": ("7",   1_815_787,   2_233_243),
    "MAD2L1": ("4",  120_055_623, 120_067_192),
}
PADDING = 5_000

def fetch_ensembl(gene):
    url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?content-type=application/json"
    try:
        d = requests.get(url, timeout=10).json()
        return str(d["seq_region_name"]), int(d["start"]), int(d["end"])
    except Exception:
        return None

gene_coords = {}
for gene, fb in FALLBACK_COORDS.items():
    result = fetch_ensembl(gene)
    gene_coords[gene] = result or fb
    c, s, e = gene_coords[gene]
    print(f"{gene:8s} {c}:{s:>12,}-{e:>12,}  largo {(e-s)/1000:.0f} kb  ({'Ensembl' if result else 'fallback'})")

In [ ]:
# Celda 8 · [Ktalyna] El filtrado ⏱️ 1–3 min
# Para cada línea del VCF: (1) descarta no-PASS; (2) descarta si no cae en los 6 genes;
# (3) guarda la variante y su zigosidad.
# Salida esperada: 1,075 variantes.
interval_idx = {}
for gene, (chrom, start, end) in gene_coords.items():
    interval_idx.setdefault(chrom, []).append((start - PADDING, end + PADDING, gene))

def gene_at(chrom, pos):
    for s, e, g in interval_idx.get(chrom, []):
        if s <= pos <= e:
            return g
    return None

def classify_zygosity(gt):
    alleles = gt.replace("|", "/").split("/")
    if "." in alleles: return "MISSING"
    if set(alleles) == {"0"}: return "HOM_REF"
    if alleles[0] == alleles[1] and alleles[0] != "0": return "HOM_ALT"
    if "0" in alleles: return "HET"
    return "OTHER"

hits = []
with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"): continue
        p = line.rstrip().split("\t")
        chrom, pos_s, vid, ref, alt, qual, filt, info, fmt = p[:9]
        sample = p[9] if len(p) > 9 else ""
        if filt != "PASS": continue
        pos = int(pos_s)
        gene = gene_at(chrom, pos)
        if not gene: continue
        gt_dict = dict(zip(fmt.split(":"), sample.split(":")))
        gt = gt_dict.get("GT", ".")
        hits.append({
            "gene": gene, "chrom": chrom, "pos": pos, "id": vid, "ref": ref, "alt": alt,
            "qual": float(qual),
            "type": "SNV" if len(ref) == 1 and len(alt) == 1 else ("DEL" if len(ref) > len(alt) else "INS"),
            "zygosity": classify_zygosity(gt), "GT": gt,
            "DP": int(gt_dict.get("DP", 0) or 0), "GQ": int(gt_dict.get("GQ", 0) or 0),
            "AD": gt_dict.get("AD", "."),
        })

df = pd.DataFrame(hits)
print(f"Total variantes PASS en genes candidatos: {len(df):,}")
df.head()

In [ ]:
# Celda 9 · [Ktalyna] Resumen por gen
# ⚠️ Más variantes ≠ más dañado: MAD1L1 tiene más porque es el gen más largo.
print(pd.crosstab(df["gene"], df["zygosity"]))

In [ ]:
# Celda 10 · [Ktalyna] Homocigotas en los genes principales
# MVA es recesiva → hacen falta las dos copias dañadas → empezar por las homocigotas (1/1).
# Salida esperada: 10 (7 en BUB1B, 3 en CEP57). Todas tienen ID rs... (conocidas en dbSNP).
primary_genes = ["BUB1B", "CEP57", "TRIP13"]
df_hom_primary = df[(df["gene"].isin(primary_genes)) & (df["zygosity"] == "HOM_ALT")] \
                   .sort_values(["gene", "pos"]).reset_index(drop=True)
print(f"Homocigotas en BUB1B / CEP57 / TRIP13: {len(df_hom_primary)}")
df_hom_primary[["gene", "chrom", "pos", "id", "ref", "alt", "type", "DP", "GQ", "AD"]]

In [ ]:
# Celda 11 · [Agregado] ¿Qué tan comunes son esas homocigotas? (gnomAD via myvariant.info)
# gnomAD = frecuencias en >800.000 personas. Una causa de enfermedad rara debe estar en < 1%.
# Salida esperada: entre 15% y 99.9% → TODAS son comunes → ninguna puede ser la causa.

def hgvs_id(chrom, pos, ref, alt):
    c = f"chr{chrom}"
    if len(ref) == len(alt) == 1:
        return f"{c}:g.{pos}{ref}>{alt}"
    if len(alt) == 1 and ref[0] == alt[0]:
        s, e = pos + 1, pos + len(ref) - 1
        return f"{c}:g.{s}del" if s == e else f"{c}:g.{s}_{e}del"
    if len(ref) == 1 and alt[0] == ref:
        return f"{c}:g.{pos}_{pos+1}ins{alt[1:]}"
    return f"{c}:g.{pos}_{pos+len(ref)-1}delins{alt}"

h = df_hom_primary.copy()
h["hgvs"] = [hgvs_id(r.chrom, r.pos, r.ref, r.alt) for r in h.itertuples()]
res = requests.post(
    "https://myvariant.info/v1/variant?assembly=hg38",
    data={"ids": ",".join(h.hgvs), "fields": "gnomad_genome.af.af"},
    timeout=60,
).json()
af = {x["query"]: x.get("gnomad_genome", {}).get("af", {}).get("af") for x in res if not x.get("notfound")}
h["% población"] = h.hgvs.map(af).mul(100).round(1)
h[["gene", "pos", "id", "ref", "alt", "% población"]]

In [ ]:
# Celda 12 · [Agregado, sobre una idea de Ktalyna] ¿Hay un ROH en MAD1L1?
#
# Ktalyna interpretó las ~200 homocigotas de MAD1L1 como un ROH (posible consanguinidad).
# Corrección: en un ROH verdadero NO puede haber heterozigotas. Aquí sí las hay.
# Resultado: heterocigotas (azul) distribuidas por todo el gen; 23% homocigotas vs 39%
# del promedio genómico → NO hay ROH. Los "418 kb" eran el tamaño del gen.
# La hipótesis de consanguinidad queda sin evidencia en este locus.

mad = df[df.gene == "MAD1L1"]
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.scatter(mad[mad.zygosity=="HOM_ALT"].pos/1e6, [1]*sum(mad.zygosity=="HOM_ALT"),
           s=8, color="#C44E52", label="HOM_ALT")
ax.scatter(mad[mad.zygosity=="HET"].pos/1e6,     [0]*sum(mad.zygosity=="HET"),
           s=8, color="#4C72B0", alpha=0.4, label="HET")
ax.set_yticks([0, 1])
ax.set_yticklabels(["HET (azul)", "HOM (rojo)"])
ax.set_xlabel("Posición en cromosoma 7 (Mb)")
ax.set_title("MAD1L1 — HET y HOM_ALT entremezcladas → sin evidencia de ROH")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("results/mad1l1_no_roh.png", dpi=150, bbox_inches="tight")
plt.show()

z = mad.zygosity.value_counts()
print(z.to_string())
print(f"% homocigotas en MAD1L1: {z.get('HOM_ALT',0)/(z.get('HOM_ALT',0)+z.get('HET',1)):.0%}")
print("(promedio de todo el genoma: 39% — sin diferencia significativa)")

In [ ]:
# Celda 13 · [Ktalyna] Heterocigotas: la otra forma de tener las dos copias dañadas
#
# Heterocigoto compuesto: dos variantes distintas, una en cada copia del gen.
# BUB1B tiene 9; tres sin ID (rs...) entre ellas 15:40209701 y 15:40220612
# → las que en el notebook 04 resultaron ser la causa probable
#   (p.Leu737Ter y p.Asn1002Lys).
# En TRIP13, dos con DP = 7 (poca evidencia, probable ruido de secuenciación).

for gene in ["BUB1B", "CEP57", "TRIP13"]:
    het = df[(df["gene"] == gene) & (df["zygosity"] == "HET")]
    print(f"\n{gene} — heterocigotas: {len(het)}")
    display(het[["chrom", "pos", "id", "ref", "alt", "DP", "GQ", "AD"]].reset_index(drop=True))

## PARTE 3 — Ampliación a los 201 genes del equipo [Ampliación]

Misma lógica que la Celda 8 de Ktalyna. Cambia de dónde salen los genes:
- La lista está en `config/genes_candidatos.txt` (editable por cualquiera del equipo, sin tocar código).
- `scripts/resolve_genes.py` traduce nombres al oficial y busca las coordenadas en MyGene.info.

In [ ]:
# Celda 14 · Cargar los 201 genes con sus coordenadas
# Si editas la lista, vuelve a correr esta celda.
# Salida esperada: 197 genes con coordenadas (4 no existen: MOXN1, RASRL1, TLX1-AS, UPP2-AS1).
import sys
import subprocess
subprocess.run([sys.executable, "scripts/resolve_genes.py"], check=True)

genes = pd.read_csv("results/genes_candidatos_grch38.tsv", sep="\t", dtype={"chrom": str})
genes = genes[genes.chrom.notna()]
print(f"Genes con coordenadas: {len(genes)}")
genes.head()

In [ ]:
# Celda 15 · El filtrado de Ktalyna, con 201 genes ⏱️ 1–3 min
# Requiere que classify_zygosity esté definida (Celda 8).
# Salida esperada: ~19,099 variantes en 195 genes.
import bisect

PADDING = 5_000
idx = {}
for g in genes.itertuples():
    idx.setdefault(g.chrom, []).append((int(g.start) - PADDING, int(g.end) + PADDING, g))
for c in idx:
    idx[c].sort(key=lambda t: t[0])
starts = {c: [s for s, _, _ in iv] for c, iv in idx.items()}

def genes_at(chrom, pos):
    if chrom not in idx:
        return []
    i = bisect.bisect_right(starts[chrom], pos)
    return [g for s, e, g in idx[chrom][max(0, i - 50):i] if s <= pos <= e]

hits201 = []
with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"): continue
        p = line.rstrip("\n").split("\t")
        if p[6] != "PASS": continue
        pos = int(p[1])
        found = genes_at(p[0], pos)
        if not found: continue
        d = dict(zip(p[8].split(":"), p[9].split(":")))
        gt = d.get("GT", ".")
        for alt in p[4].split(","):
            for g in found:
                hits201.append({
                    "gene": g.simbolo, "categoria": g.categoria,
                    "chrom": p[0], "pos": pos,
                    "id": p[2], "ref": p[3], "alt": alt,
                    "zygosity": classify_zygosity(gt), "GT": gt,
                    "DP": int(d.get("DP", 0) or 0), "GQ": int(d.get("GQ", 0) or 0),
                    "AD": d.get("AD", "."),
                    "ubicacion": "dentro_gen" if int(g.start) <= pos <= int(g.end) else "margen_5kb",
                })

df201 = pd.DataFrame(hits201)
print(f"Variantes PASS en los genes candidatos: {len(df201):,}")
print(f"Genes con al menos una variante: {df201.gene.nunique()}")
df201.to_csv("results/variants_201_genes.tsv", sep="\t", index=False)
df201.head()

In [ ]:
# Celda 16 · Resumen por categoría y por gen
# Los que más tienen son los más largos (SYNE1, MAD1L1, SYNE2).
# BUB1B debe tener las mismas 16 que en la Parte 2.
print(df201.groupby("categoria").size().sort_values(ascending=False).to_string())
pd.crosstab(df201.gene, df201.zygosity) \
  .assign(total=lambda t: t.sum(axis=1)) \
  .sort_values("total", ascending=False).head(20)

## Lo que sigue (notebook 04 en adelante)

1. **Rareza — gnomAD:** quedarse con variantes en < 1% de la población (19,099 → ~669).
2. **Impacto — VEP + CADD + ClinVar:** ¿daña la proteína? (669 → ~7).
3. **Dos copias afectadas — patrón recesivo** → **BUB1B: p.Leu737Ter + p.Asn1002Lys**.
4. **Sensibilidad** — revisar variantes no-PASS en genes candidatos.
5. Mecanismo (Fase 3) → Fármacos (Fase 4, DGIdb) → Entrega (Fase 5).